# BAO 단독 피팅


In [ ]:
from pathlib import Path

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "requirements.txt").is_file() and (p / "data").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("project_cos 폴더에서 노트북을 실행하세요.")
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
for name in ("figures", "tables", "runs"):
    (RESULTS_DIR / name).mkdir(parents=True, exist_ok=True)


In [ ]:
import numpy as np
import pandas as pd

bao_dir = DATA_DIR / "bao"

bao_df = pd.read_csv(
    bao_dir / "desi_gaussian_bao_ALL_GCcomb_mean.txt",
    sep=r"\s+",
    comment="#",
    names=["z", "value", "quantity"],
)
bao_cov = np.loadtxt(
    bao_dir / "desi_gaussian_bao_ALL_GCcomb_cov.txt"
)

bao_z = bao_df["z"].to_numpy()
bao_y = bao_df["value"].to_numpy()
bao_quantity = bao_df["quantity"].to_numpy()

assert bao_cov.shape == (len(bao_y), len(bao_y))
assert np.allclose(bao_cov, bao_cov.T)

# A = c / (H0 * rd)
def bao_model_shape(om):
    grid = np.linspace(0, bao_z.max(), 5000)
    inverse_E = 1 / np.sqrt(om * (1 + grid)**3 + 1 - om)

    area = (inverse_E[:-1] + inverse_E[1:]) / 2 * np.diff(grid)
    integral = np.r_[0, np.cumsum(area)]

    dm = np.interp(bao_z, grid, integral)
    dh = 1 / np.sqrt(om * (1 + bao_z)**3 + 1 - om)
    dv = np.cbrt(bao_z * dm**2 * dh)

    predictions = {
        "DM_over_rs": dm,
        "DH_over_rs": dh,
        "DV_over_rs": dv,
    }
    return np.array([
        predictions[name][i]
        for i, name in enumerate(bao_quantity)
    ])

bao_omega = np.linspace(0.01, 0.99, 981)
bao_shapes = np.column_stack([
    bao_model_shape(om) for om in bao_omega
])

bao_L = np.linalg.cholesky(bao_cov)
bao_weighted = np.linalg.solve(
    bao_L, np.column_stack((bao_y, bao_shapes))
)
bao_y_weighted = bao_weighted[:, 0]
bao_shapes_weighted = bao_weighted[:, 1:]

bao_A = (
    bao_shapes_weighted.T @ bao_y_weighted
) / np.sum(bao_shapes_weighted**2, axis=0)

bao_residuals = (
    bao_y_weighted[:, None] - bao_shapes_weighted * bao_A
)
bao_chi2 = np.sum(bao_residuals**2, axis=0)

bao_best = np.argmin(bao_chi2)
bao_interval = bao_omega[bao_chi2 - bao_chi2.min() <= 1]

print(f"BAO 최적 Ωm = {bao_omega[bao_best]:.3f}")
print(
    f"근사 68% 구간 = "
    f"{bao_interval.min():.3f} ~ {bao_interval.max():.3f}"
)
print(f"최적 A = c/(H0·rd) = {bao_A[bao_best]:.3f}")
print(
    f"최소 χ² / 자유도 = "
    f"{bao_chi2[bao_best]:.2f} / {len(bao_y) - 2}"
)


BAO 최적 Ωm = 0.297
근사 68% 구간 = 0.289 ~ 0.306
최적 A = c/(H0·rd) = 29.514
최소 χ² / 자유도 = 10.27 / 11
